# Relational KD (similarity-preserving) — warm-start

Same warm-start regime as `run_forward_warm_distill`, but the feature term is
**relational** instead of value-matching: it matches the frame-frame similarity
structure of the teacher (Tung & Mori, ICCV 2019), enabled by `feat_mode: rkd_sp`
in `rkd_warm_fd_cmkd.yaml`. Tests whether transferring *structure* (rather than
feature values) beats the neutral FD-CMKD transfer.

Compare the result against the matched control (39.70 dev / 40.45 test).
Run All. The notebook auto-`chdir`s to `code/distillation`, so it works from here.

In [ ]:
# Run from code/distillation/ no matter where the kernel was launched.
import os
if not os.path.exists('fd_cmkd.py'):
    _cwd = os.getcwd()
    for _k in range(0, 6):
        _base = os.path.abspath(os.path.join(_cwd, *(['..'] * _k)))
        _cand = os.path.join(_base, 'code', 'distillation')
        if os.path.exists(os.path.join(_cand, 'fd_cmkd.py')):
            os.chdir(_cand); break
        if os.path.exists(os.path.join(_base, 'fd_cmkd.py')):
            os.chdir(_base); break
print('working dir:', os.getcwd())

import os
RUN_NAME  = 'forward_warm_rkd'   # RKD relazionale (non sovrascrive gli altri run)
OVERWRITE = False
DO_TRAIN  = True

RUN_DIR = os.path.join('../../runs', RUN_NAME)
if os.path.exists(RUN_DIR) and not OVERWRITE:
    _n = 2
    while os.path.exists('%s_%03d' % (RUN_DIR, _n)):
        _n += 1
    RUN_DIR = '%s_%03d' % (RUN_DIR, _n)
print('this run writes to:', os.path.normpath(os.path.abspath(RUN_DIR)))
EVAL_CKPT = 'mine'

In [ ]:
import sys, yaml
import torch
# Questo ambiente ha cuDNN difettoso (CUDNN_STATUS_NOT_INITIALIZED su Conv1d/LSTM);
# il teacher stesso e' allenato con cuDNN off (csrl_skeleton/config.py). Disattivalo
# per l'intero kernel: i layer usano il kernel CUDA nativo (piu' lento, ma funziona).
torch.backends.cudnn.enabled = False

SIGNFORMER_DIR = os.path.abspath('../signformer')   # for `import main.*`
NB_DIR = os.path.abspath('.')                        # for `import fd_cmkd*`
for _p in (SIGNFORMER_DIR, NB_DIR):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import main.training as mt
from fd_cmkd_trainer import train_fd_cmkd
from fd_cmkd import load_teacher_feats
print('imports OK | cudnn enabled:', torch.backends.cudnn.enabled)

In [ ]:
# ── ①  Teacher features check + estrazione (fai tutto da qui, niente terminale) ──
# La distillazione legge le FEATURE del teacher (skeleton_feats_133/*.pkl), non il
# checkpoint. Questa cella controlla se ci sono; se mancano, le estrae dal teacher.
import yaml

_dc = yaml.safe_load(open('warm_start_fd_cmkd.yaml'))['training']['distillation']
FEAT_DIR = _dc['teacher_feats_dir']                    # ../../dataset/features/skeleton_feats_133
_train_pkl = os.path.join(FEAT_DIR, 'train.pkl')

# EDIT questi due percorsi (relativi a code/distillation, o assoluti):
TEACHER_CKPT = '../../runs/run133/tssi133_gcn_best.pt'      # il tuo teacher 41.75
DATA_133     = '../../dataset/pose/phoenix2014t_133kp'      # cartella coi Phoenix-2014T.{train,dev,test}

if os.path.exists(_train_pkl):
    print('OK — teacher features già presenti in', FEAT_DIR, '(skip estrazione)')
else:
    print('teacher features assenti → estraggo dal checkpoint (~15-30 min)...')
    assert os.path.exists(TEACHER_CKPT), 'checkpoint teacher non trovato: ' + TEACHER_CKPT
    os.environ['TEACHER_CKPT']  = os.path.abspath(TEACHER_CKPT)
    os.environ['MSKA_DATA_DIR'] = os.path.abspath(DATA_133)
    get_ipython().system('python extract_skeleton_feats.py')
    assert os.path.exists(_train_pkl), 'estrazione fallita, manca ' + _train_pkl
    print('feature estratte →', FEAT_DIR)

In [ ]:
# Sanity checks: data + teacher features present; RKD config with feat_mode=rkd_sp.
BASE_CFG = 'rkd_warm_fd_cmkd.yaml'
assert os.path.exists(BASE_CFG), 'missing config: ' + BASE_CFG

cfg = yaml.safe_load(open(BASE_CFG))
dc = cfg['training']['distillation']
assert dc.get('feat_mode') == 'rkd_sp', 'this config must set feat_mode: rkd_sp'

i3d = os.path.join(cfg['data']['data_path'], cfg['data']['train'])
assert os.path.exists(i3d), 'missing I3D features: ' + i3d

tfp = os.path.join(dc['teacher_feats_dir'], 'train.pkl')
assert os.path.exists(tfp), 'missing teacher features: ' + tfp
_t = load_teacher_feats(tfp)

assert 'load_model' in cfg['training'], 'a warm-start run must load the baseline checkpoint'
print('feat_mode          :', dc['feat_mode'])
print('teacher features   : %d videos, dim %d' % (len(_t), next(iter(_t.values())).shape[1]))
print('warm-start         : loads', cfg['training']['load_model'])

In [ ]:
# Point the config at RUN_DIR, honour OVERWRITE, write the resolved config there.
# The committed warm_start_fd_cmkd.yaml is never modified.
# RUN_DIR is a container. The trainer writes into RUN_DIR/final, which it must
# be free to create fresh: make_model_dir refuses a directory that already
# exists. Writing the config into RUN_DIR (not into MODEL_DIR) keeps them apart.
MODEL_DIR = os.path.join(RUN_DIR, 'final')
cfg['training']['model_dir'] = MODEL_DIR
cfg['training']['overwrite'] = OVERWRITE

os.makedirs(RUN_DIR, exist_ok=True)
ACTIVE_CFG = os.path.join(RUN_DIR, 'warm_start_fd_cmkd.yaml')
with open(ACTIVE_CFG, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

tr = cfg['training']
print('active config ->', ACTIVE_CFG)
print('lr=%.3e  wd=%.3e  batch=%d  patience=%d' % (tr['learning_rate'], tr['weight_decay'], tr['batch_size'], tr['early_stopping_patience']))
print('lambda_feat=%.3f  lambda_align=%.3f  w_high=%.3f  warmup=%d' % (dc['lambda_feat'], dc['lambda_align'], dc['high_w'], dc['distill_warmup_steps']))

## Optional: matched learning-rate search

The baseline was selected over a small Optuna search. For the comparison to be fair,
the from-scratch distilled arm deserves a search of the **same budget** over the
**same learning-rate range**. Turn it on to run it; each trial writes to a temporary
directory under `RUN_DIR`, and the best learning rate is kept.

Leave it off (`USE_OPTUNA = False`) to train a single run at the baseline learning
rate -- a quick check that the pipeline works, but not, on its own, evidence.

In [ ]:
import copy, shutil, gc

USE_OPTUNA    = False   # True re-runs the matched search (hours)
OPTUNA_TRIALS = 6       # match the baseline budget
LR_RANGE      = (1.5e-4, 6e-4)   # a neighbourhood of the baseline 3.12e-4

if USE_OPTUNA and DO_TRAIN:
    import torch
    try:
        import optuna
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'optuna'])
        import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    _log = []
    def _objective(trial):
        lr = trial.suggest_float('learning_rate', LR_RANGE[0], LR_RANGE[1], log=True)
        ct = copy.deepcopy(cfg)
        ct['training']['learning_rate'] = float(lr)
        td = os.path.join(RUN_DIR, 'optuna', 'trial_%02d' % trial.number)
        ct['training']['model_dir'] = td
        ct['training']['overwrite'] = True
        # config beside the trial dir, not inside: make_model_dir wipes the
        # dir (overwrite=True) and would delete a config written into it.
        os.makedirs(os.path.dirname(td), exist_ok=True)
        cf = td + '.yaml'
        with open(cf, 'w') as f:
            yaml.safe_dump(ct, f, sort_keys=False)
        train_fd_cmkd(cf)
        wers = []
        vf = os.path.join(ct['training']['model_dir'], 'validations.txt')
        if os.path.exists(vf):
            for line in open(vf):
                toks = line.replace(':', ' ').split()
                if 'WER' in toks:
                    try:
                        wers.append(float(toks[toks.index('WER') + 1]))
                    except (ValueError, IndexError):
                        pass
        best = min(wers) if wers else 100.0
        _log.append((trial.number, lr, best))
        print('  trial %d: lr=%.3e -> dev WER %.2f' % (trial.number, lr, best))
        shutil.rmtree(td, ignore_errors=True)
        if os.path.exists(cf):
            os.remove(cf)
        gc.collect(); torch.cuda.empty_cache()
        return best

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(_objective, n_trials=OPTUNA_TRIALS)
    print('best lr:', study.best_params, '-> dev WER', round(study.best_value, 2))
    cfg['training']['learning_rate'] = float(study.best_params['learning_rate'])
    with open(ACTIVE_CFG, 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    print('ACTIVE_CFG updated with the searched learning rate.')
else:
    print('Optuna search disabled (or DO_TRAIN=False) -- using the config lr.')

In [ ]:
# Train from scratch. This is the long cell.
if DO_TRAIN:
    train_fd_cmkd(ACTIVE_CFG)
else:
    print('DO_TRAIN = False -> skipping training. Set it True to run from scratch.')

In [ ]:
# Curves: train loss and validation loss (left), dev WER (right).
import re
import matplotlib.pyplot as plt
from main.helpers import parse_validations

MDIR = os.path.join(RUN_DIR, 'final')
val = parse_validations(MDIR)   # steps, recognition_loss (dev), wer, lr

# Train loss is logged per step in train.log, not in validations.txt.
tr_steps, tr_loss = [], []
logp = os.path.join(MDIR, 'train.log')
if os.path.exists(logp):
    for line in open(logp, encoding='utf-8', errors='ignore'):
        ms = re.search(r'Step:\s*(\d+)', line)
        ml = re.search(r'Batch Recognition Loss:\s*([\d.]+)', line)
        if ms and ml:
            tr_steps.append(int(ms.group(1)))
            tr_loss.append(float(ml.group(1)))

if not val['steps'] and not tr_steps:
    print('no logs yet in', MDIR, '(train first, DO_TRAIN = True)')
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

    if tr_steps:
        ax1.plot(tr_steps, tr_loss, lw=0.8, alpha=0.5, color='tab:blue', label='train loss')
    if val['steps']:
        ax1.plot(val['steps'], val['recognition_loss'], marker='o', ms=3,
                 color='tab:red', label='val loss')
    ax1.set_xlabel('step'); ax1.set_ylabel('CTC loss'); ax1.set_title('Loss')
    ax1.legend(); ax1.grid(alpha=0.3)

    if val['steps']:
        ax2.plot(val['steps'], val['wer'], marker='o', ms=3, color='crimson', label='dev WER')
        ax2.axhline(39.54, ls='--', c='gray', label='baseline (greedy) 39.54')
        good = [(s, w) for s, w in zip(val['steps'], val['wer']) if w == w]
        if good:
            bs, bw = min(good, key=lambda t: t[1])
            ax2.scatter([bs], [bw], color='black', zorder=5, label='best %.2f' % bw)
    ax2.set_xlabel('step'); ax2.set_ylabel('dev WER (%)'); ax2.set_title('Dev WER')
    ax2.legend(); ax2.grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig(os.path.join(MDIR, 'training_curves.png'), dpi=120)
    plt.show()

    if val['steps']:
        good = [(s, w) for s, w in zip(val['steps'], val['wer']) if w == w]
        if good:
            bs, bw = min(good, key=lambda t: t[1])
            print('best dev WER : %.2f at step %d' % (bw, bs))
        vl = [x for x in val['recognition_loss'] if x == x]
        if vl:
            print('last val loss: %.3f' % vl[-1])
    if tr_loss:
        print('last train loss: %.3f' % tr_loss[-1])


In [ ]:
# Evaluate the best-on-dev checkpoint and lay the numbers next to the baseline.
best_ckpt = os.path.join(RUN_DIR, 'final', 'best.ckpt')
if not os.path.exists(best_ckpt):
    print('no best.ckpt yet -- train first (DO_TRAIN = True).')
else:
    mt.test(ACTIVE_CFG, ckpt=best_ckpt, output_path=os.path.join(RUN_DIR, 'final', 'test_output'))
    print()
    print('Baseline, from scratch, CTC only (sign.yaml):')
    print('   dev  39.54 greedy / 39.35 beam-4      test  41.53')
    print('This run, from scratch, CTC + FD-CMKD:')
    print('   see the [DEV] and [TEST] WER printed above')
    print()
    print('Reading the result:')
    print(' - below baseline beyond the margin: warm-start was hiding a real transfer.')
    print(' - within the margin: the last alternative explanation is ruled out.')
    print(' - above baseline: a weak teacher shaping the representation transfers its confusion.')